# 09. Engagement Twitter

Module descriptif orienté publication: comment l'engagement varie selon les blocs, la tonalité (`stance_v3`) et les cadrages.

**Prérequis**: le corpus doit contenir une colonne `engagement` ou des colonnes d'interactions (`retweets`, `likes`, `replies`, `quotes`).

**Portée**: les résultats restent descriptifs; ils ne testent pas une causalité entre radicalité et engagement.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import CORPUS_V3, FIGURES_DIR, BLOC_ORDER, BLOC_COLORS

In [ ]:
df = pd.read_parquet(CORPUS_V3)
arena_col = "arena" if "arena" in df.columns else None
df_tw = df[df[arena_col] == "Twitter"] if arena_col else pd.DataFrame()

engagement_cols = [c for c in ["retweet_count", "like_count", "reply_count", "favorite_count", "engagement"] if c in df.columns]
if not engagement_cols:
    engagement_cols = [c for c in df.columns if "count" in c.lower() or "like" in c.lower() or "retweet" in c.lower()]

print("Colonnes engagement trouvées:", engagement_cols or "Aucune")
print("Tweets (arena=Twitter):", len(df_tw) if len(df_tw) else "N/A (colonne arena absente)")

Colonnes engagement trouvées: ['engagement']
Tweets (arena=Twitter): 9135


## 1) Engagement agrégé par bloc

Comparer les distributions d'engagement par bloc permet de distinguer volume de production et capacité de relais.

In [ ]:
if len(df_tw) > 0 and engagement_cols:
    eng_col = engagement_cols[0]
    if "engagement" in engagement_cols:
        eng_col = "engagement"
    else:
        df_tw = df_tw.copy()
        df_tw["engagement"] = df_tw[engagement_cols].fillna(0).sum(axis=1)
        eng_col = "engagement"

    df_tw = df_tw[df_tw["bloc"].isin(BLOC_ORDER)]
    df_tw["log_engagement"] = np.log1p(df_tw[eng_col])
    agg = df_tw.groupby("bloc").agg(mean_log=("log_engagement", "mean"), n=("log_engagement", "count")).reindex(BLOC_ORDER)
    print(agg.to_string())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(BLOC_ORDER)), agg["mean_log"], color=[BLOC_COLORS[b] for b in BLOC_ORDER])
    ax.set_xticks(range(len(BLOC_ORDER)))
    ax.set_xticklabels([b.replace(" ", "\n") for b in BLOC_ORDER])
    ax.set_ylabel("log(1 + engagement) moyen")
    ax.set_title("Engagement Twitter par bloc")
    plt.tight_layout()
    FIGURES_DIR.mkdir(exist_ok=True)
    plt.savefig(FIGURES_DIR / "fig50_engagement_par_bloc.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("→ fig50_engagement_par_bloc.png")
else:
    print("Données d'engagement absentes. Ajouter retweet_count, like_count, etc. au corpus pour activer cette analyse.")

                   mean_log     n
bloc                             
Gauche radicale    5.539016  6227
Gauche moderee     4.615323   814
Centre / Majorite  4.308860   944
Droite             4.263186  1150
→ fig50_engagement_par_bloc.png


## 2) Intensité de stance et engagement

Question descriptive: les tweets à `|stance_v3|` élevé sont-ils associés à un engagement plus fort?

In [ ]:
if len(df_tw) > 0 and "log_engagement" in df_tw.columns and "stance_v3" in df_tw.columns:
    df_tw["abs_stance"] = df_tw["stance_v3"].abs()
    r = df_tw["abs_stance"].corr(df_tw["log_engagement"])
    print(f"Corrélation descriptive |stance| ↔ log(engagement): r = {r:.3f}")
    if abs(r) > 0.05:
        print("→ Association observée: les tweets plus intenses tendent à être davantage relayés.")
    else:
        print("→ Association faible: l'intensité de stance explique peu l'engagement observé.")
    print("→ Prudence: corrélation descriptive, pas d'inférence causale.")

Corrélation |stance| ↔ log(engagement) : r = 0.245
→ La radicalité est associée à un engagement différencié.
